In [1]:
import pandas as pd

# Load Tranco list (no header: column 0 = rank, column 1 = domain)
tranco = pd.read_csv('../dataset/tranco_top.csv', header=None, names=['rank', 'domain'])

print("Total domains:", len(tranco))
print("\nFirst 10:")
print(tranco.head(10).to_string())
print("\nLast 5:")
print(tranco.tail(5).to_string())

Total domains: 1000000

First 10:
   rank            domain
0     1        google.com
1     2  gtld-servers.net
2     3    cloudflare.com
3     4       gstatic.com
4     5      facebook.com
5     6     microsoft.com
6     7    googleapis.com
7     8       youtube.com
8     9     amazonaws.com
9    10         apple.com

Last 5:
           rank                        domain
999995   999996                altyzhub9.site
999996   999997                alus4dpro.site
999997   999998             alutore-masser.at
999998   999999  alvarezroofingcontractor.com
999999  1000000             alvespneus.com.br


In [4]:
#Take the top 25,000 popular legitimate domains
tranco_benign = tranco.head(25000).copy()

#Keep only domain column, label all as benign (0)
tranco_benign = tranco_benign[['domain']].rename(columns={'domain': 'url'})
tranco_benign['label'] = 0

print("Augmentation samples:", len(tranco_benign))
print("\nSample:")
print(tranco_benign.head(10).to_string())
print("\nLabel check:")
print(tranco_benign['label'].value_counts())

Augmentation samples: 25000

Sample:
                url  label
0        google.com      0
1  gtld-servers.net      0
2    cloudflare.com      0
3       gstatic.com      0
4      facebook.com      0
5     microsoft.com      0
6    googleapis.com      0
7       youtube.com      0
8     amazonaws.com      0
9         apple.com      0

Label check:
label
0    25000
Name: count, dtype: int64


In [5]:
import sys
sys.path.append('..')
from src.url_features import extract_url_features

print("Extracting URL features from Tranco Domains...")
tranco_features = tranco_benign['url'].apply(extract_url_features)
tranco_features_df = pd.DataFrame(tranco_features.tolist())

#Adding the label 
tranco_features_df['label'] = tranco_benign['label'].values

print("Done! Shape:", tranco_features_df.shape)
print("\nSample (note num_slashes and path_length should be 0 for bare domains):")
print(tranco_features_df[['url_length', 'domain_length', 'num_slashes', 'path_length', 'num_subdomains', 'label']].head(10).to_string())

Extracting URL features from Tranco Domains...
Done! Shape: (25000, 30)

Sample (note num_slashes and path_length should be 0 for bare domains):
   url_length  domain_length  num_slashes  path_length  num_subdomains  label
0          10             10            0            0               0      0
1          16             16            0            0               0      0
2          14             14            0            0               0      0
3          11             11            0            0               0      0
4          12             12            0            0               0      0
5          13             13            0            0               0      0
6          14             14            0            0               0      0
7          11             11            0            0               0      0
8          13             13            0            0               0      0
9           9              9            0            0               0     

In [6]:
# Load existing URL features
df_existing = pd.read_csv('../dataset/url_features.csv')
print("Existing dataset shape:", df_existing.shape)
print("Existing label distribution:\n", df_existing['label'].value_counts())

# Make sure columns match before combining
print("\nColumns match:", list(df_existing.columns) == list(tranco_features_df.columns))

# Combine
df_augmented = pd.concat([df_existing, tranco_features_df], ignore_index=True)

print("\nAugmented dataset shape:", df_augmented.shape)
print("New label distribution:\n", df_augmented['label'].value_counts())

# Save
df_augmented.to_csv('../dataset/url_features_augmented.csv', index=False)
print("\nSaved as url_features_augmented.csv!")

Existing dataset shape: (446176, 30)
Existing label distribution:
 label
0    223088
1    223088
Name: count, dtype: int64

Columns match: True

Augmented dataset shape: (471176, 30)
New label distribution:
 label
0    248088
1    223088
Name: count, dtype: int64

Saved as url_features_augmented.csv!


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

# Load augmented dataset
df_aug = pd.read_csv('../dataset/url_features_augmented.csv')

X = df_aug.drop('label', axis=1)
y = df_aug['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training augmented URL-only model...")
rf_url_aug = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_url_aug.fit(X_train, y_train)

y_pred = rf_url_aug.predict(X_test)
print("\n=== Augmented URL-Only Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

# Test google.com specifically
print("\n=== Testing google.com ===")
test_feats = extract_url_features('google.com')
test_df = pd.DataFrame([test_feats])
pred = rf_url_aug.predict(test_df)[0]
prob = rf_url_aug.predict_proba(test_df)[0]
print("Prediction:", "Malicious" if pred == 1 else "Benign")
print("Probability [benign, malicious]:", prob)

Training augmented URL-only model...

=== Augmented URL-Only Model Performance ===
Accuracy:  0.8837
Precision: 0.8793
Recall:    0.8744
F1 Score:  0.8769

=== Testing google.com ===
Prediction: Benign
Probability [benign, malicious]: [0.85275321 0.14724679]


In [8]:
joblib.dump(rf_url_aug, '../model/url_only_model.pkl')
print("Augmented URL-only model saved!")

Augmented URL-only model saved!


In [9]:
import shap
import pickle

print("Regenerating URL SHAP explainer for the augmented model...")
explainer_url = shap.TreeExplainer(rf_url_aug)

with open('../model/shap_explainer_url.pkl', 'wb') as f:
    pickle.dump(explainer_url, f)

print("URL SHAP explainer regenerated and saved!")

Regenerating URL SHAP explainer for the augmented model...
URL SHAP explainer regenerated and saved!
